# Visualizing Hierarchical Data: Treemaps and Sunburst Charts

**A step-by-step practical lesson using Python, Pandas and Plotly**

## Learning objectives

By the end of this notebook, you should be able to:

1. Recognize hierarchical data and identify parent, child and leaf nodes.
2. Generate and inspect a realistic multi-level dataset.
3. Explain why **rectangle area in a treemap is proportional to the value**.
4. Build and interpret interactive treemaps and sunburst charts.
5. Customize labels, colours, hover information, layout and depth.
6. Decide which visualization is appropriate for a given analytical question.

> **Dataset note:** The sales data in this notebook are synthetic teaching data. They are generated for practice and do not represent a real company.


## 1. What is hierarchical data?

Hierarchical data is organized into levels. A node at one level can contain one or more nodes at the next level.

For this lesson, imagine an online retailer:

- **Retailer** (root)
  - **Category** (e.g., Electronics)
    - **Subcategory** (e.g., Mobiles)
      - **Product** (e.g., Smartphone A)

The relationship is a tree: one parent can have multiple children, and each child belongs to one parent.

### Important vocabulary

| Term | Meaning | Example |
|---|---|---|
| Root | Topmost node | Retailer |
| Parent | Node containing lower-level nodes | Electronics |
| Child | Node belonging to a parent | Mobiles |
| Leaf | Node with no children | Smartphone A |
| Depth / level | Position in the hierarchy | Category is level 1 |

A normal flat table can store the hierarchy in columns. The hierarchy is represented by the repeated parent labels in each row.


## 2. Import libraries

In [ ]:
# If needed, uncomment and run this line:
# %pip install pandas numpy plotly

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", None)


## 3. Generate a hierarchical sales dataset

We will generate data at the **product level**. Each row is one product, and the columns show the category path to that product.

The value column is `Sales_Lakh`. Treemaps and sunburst charts will aggregate this value upward:

\[
\text{Subcategory sales}=\sum \text{sales of its products}
\]

\[
\text{Category sales}=\sum \text{sales of its subcategories}
\]

This is called **bottom-up aggregation**.


In [ ]:
# Fixed seed makes the example reproducible
rng = np.random.default_rng(42)

catalogue = {
    "Electronics": {
        "Mobiles": ["Smartphone A", "Smartphone B", "Feature Phones"],
        "Computers": ["Laptops", "Desktops", "Tablets"],
        "Home Entertainment": ["Televisions", "Speakers", "Streaming Devices"],
    },
    "Fashion": {
        "Men": ["Shirts", "Trousers", "Footwear"],
        "Women": ["Dresses", "Tops", "Footwear"],
        "Accessories": ["Watches", "Bags", "Belts"],
    },
    "Home & Kitchen": {
        "Kitchen": ["Cookware", "Mixer Grinders", "Small Appliances"],
        "Furniture": ["Chairs", "Tables", "Storage"],
        "Home Care": ["Vacuum Cleaners", "Air Purifiers", "Cleaning Tools"],
    },
    "Sports": {
        "Fitness": ["Dumbbells", "Yoga Mats", "Treadmills"],
        "Outdoor": ["Camping Gear", "Cycles", "Hiking Equipment"],
        "Team Sports": ["Football Gear", "Cricket Gear", "Badminton Gear"],
    },
}

rows = []
for category, subcategories in catalogue.items():
    for subcategory, products in subcategories.items():
        for product in products:
            # Synthetic sales value, deliberately varied
            sales = round(float(rng.uniform(8, 85)), 2)
            units = int(rng.integers(80, 1800))
            rows.append({
                "Category": category,
                "Subcategory": subcategory,
                "Product": product,
                "Sales_Lakh": sales,
                "Units_Sold": units
            })

df = pd.DataFrame(rows)
df.head(30)


,Category,Subcategory,Product,Sales_Lakh,Units_Sold
0,Electronics,Mobiles,Smartphone A,67.59,1205
1,Electronics,Mobiles,Smartphone B,74.11,834
2,Electronics,Mobiles,Feature Phones,61.70,426
3,Electronics,Computers,Laptops,83.12,241
4,Electronics,Computers,Desktops,66.61,1314
5,Electronics,Computers,Tablets,17.86,1432
6,Electronics,Home Entertainment,Televisions,42.68,940
7,Electronics,Home Entertainment,Speakers,79.36,717
8,Electronics,Home Entertainment,Streaming Devices,57.58,772
9,Fashion,Men,Shirts,42.14,1495


In [ ]:
print("Rows:", len(df))
print("Categories:", df["Category"].nunique())
print("Subcategories:", df["Subcategory"].nunique())
print("Products:", df["Product"].nunique())
print("Total sales (₹ lakh):", round(df["Sales_Lakh"].sum(), 2))

df.describe()


Rows: 36
Categories: 4
Subcategories: 12
Products: 35
Total sales (₹ lakh): 1783.77


,Sales_Lakh,Units_Sold
count,36.000000,36.000000
mean,49.549167,923.138889
std,22.535414,483.116972
min,8.570000,196.000000
25%,29.025000,523.500000
50%,59.385000,892.500000
75%,67.677500,1332.250000
max,83.120000,1749.000000


## 4. Understand the hierarchy before plotting

Look at the first few rows. A product row contains its complete path:

`Category → Subcategory → Product`

For example, a row with `Electronics`, `Mobiles`, `Smartphone A` is a leaf product inside Mobiles, which is inside Electronics.

### Aggregate the data manually

This helps us verify the totals that the chart will display.


In [ ]:
subcategory_totals = (
    df.groupby(["Category", "Subcategory"], as_index=False)["Sales_Lakh"]
      .sum()
      .sort_values("Sales_Lakh", ascending=False)
)

category_totals = (
    df.groupby("Category", as_index=False)["Sales_Lakh"]
      .sum()
      .sort_values("Sales_Lakh", ascending=False)
)

display(category_totals)
display(subcategory_totals.head(10))


,Category,Sales_Lakh
0,Electronics,550.61
3,Sports,457.99
1,Fashion,442.60
2,Home & Kitchen,332.57


,Category,Subcategory,Sales_Lakh
9,Sports,Fitness,206.12
2,Electronics,Mobiles,203.40
3,Fashion,Accessories,188.65
1,Electronics,Home Entertainment,179.62
5,Fashion,Women,173.40
0,Electronics,Computers,167.59
10,Sports,Outdoor,160.72
8,Home & Kitchen,Kitchen,137.31
7,Home & Kitchen,Home Care,103.05
6,Home & Kitchen,Furniture,92.21


### Check the aggregation rule

The sum of all category totals should equal the sum of all product-level sales.

$$
\sum_{\text{categories}} \text{Category sales}
=
\sum_{\text{products}} \text{Product sales}
$$

In [ ]:
assert np.isclose(category_totals["Sales_Lakh"].sum(), df["Sales_Lakh"].sum())
print("Check passed: category totals add up to total product sales.")


Check passed: category totals add up to total product sales.


# Part A — Treemaps

## 5. What is a treemap?

A **treemap** represents hierarchical data as a collection of nested rectangles.

- Each rectangle represents a node in the hierarchy.
- Parent rectangles contain their child rectangles.
- The **area** of a rectangle represents a numerical value.
- Colour can encode another variable or repeat the size variable.

A treemap is useful when we want to compare the relative contribution of many categories while using screen space efficiently.


## 6. The most important idea: Rectangle area is proportional to value

Suppose three products have the following sales:

| Product | Sales |
|---|---:|
| A | 50 |
| B | 30 |
| C | 20 |
| **Total** | **100** |

If the entire treemap drawing area is 10,000 square pixels, the ideal rectangle areas will be:

- Product A: (50 / 100) × 10,000 = **5,000 pixels²**
- Product B: (30 / 100) × 10,000 = **3,000 pixels²**
- Product C: (20 / 100) × 10,000 = **2,000 pixels²**

### General rule

**Rectangle area = (Node value / Total value) × Available drawing area**

Therefore, the area ratio is:

**A : B : C = 50 : 30 : 20 = 5 : 3 : 2**

### Important distinction

It is the **area**, not the width or height alone, that is proportional to value.

Remember:

- Area = Width × Height
- A larger value produces a larger rectangle area.
- Width and height can vary depending on the treemap layout algorithm.
- Two rectangles can have the same area but different shapes.

This is why we should compare the total rectangular area rather than just the width or height.

In [ ]:
# A simple numerical demonstration
toy = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Sales": [50, 30, 20]
})

drawing_area = 10_000  # illustrative square-pixel area
toy["Share"] = toy["Sales"] / toy["Sales"].sum()
toy["Ideal_Area"] = toy["Share"] * drawing_area
toy["Area_Percent"] = toy["Share"] * 100

toy


,Product,Sales,Share,Ideal_Area,Area_Percent
0,A,50,0.5,5000.0,50.0
1,B,30,0.3,3000.0,30.0
2,C,20,0.2,2000.0,20.0


### Why do the rectangles have different shapes?

A treemap layout algorithm must pack rectangles into the available canvas. It chooses widths and heights so that:

\[
\text{Area}=\text{Width}\times\text{Height}
\]

The algorithm tries to arrange them neatly and reduce awkward, very thin rectangles. Consequently:

- A large value gets a large **area**.
- It does not necessarily get a proportionally larger width.
- Two rectangles with equal areas can have different widths and heights.

Plotly's treemap uses a squarified layout by default. Squarified layouts try to make rectangles less elongated, improving readability.


## 7. Create your first treemap

In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    title="Retail Sales Treemap: Category → Subcategory → Product"
)

fig.show()


### Read the chart

1. The largest outer group is the category with the highest total sales.
2. Inside a category, each rectangle represents a subcategory.
3. The smallest rectangles represent individual products.
4. Click a rectangle to zoom into that part of the hierarchy; use the chart's navigation controls to move back up.

The values are aggregated from product rows. For example, the Electronics parent value is the sum of all Electronics products.


## 8. Treemap: show values, percentages and hover details

A useful chart should not make the audience guess what a rectangle means. We can customize:
- text shown inside rectangles;
- colour;
- hover information;
- title and margins.


In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    color="Sales_Lakh",
    color_continuous_scale="Blues",
    title="Retail Sales by Category, Subcategory and Product",
)

fig.update_traces(
    textinfo="label+value+percent parent",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Sales: ₹%{value:.2f} lakh<br>"
        "Share of parent: %{percentParent:.1%}<br>"
        "Share of total: %{percentRoot:.1%}"
        "<extra></extra>"
    )
)

fig.update_layout(
    title_x=0.5,
    margin=dict(t=70, l=20, r=20, b=20),
    coloraxis_colorbar=dict(title="Sales<br>(₹ lakh)")
)

fig.show()


### Understanding the percentages

- **Share of parent (`percent parent`)**: a node's value divided by its immediate parent's value.
- **Share of total (`percent root`)**: a node's value divided by the root total.

For example, a product may contribute 25% of its subcategory but only 2% of the entire retailer's sales. Both percentages answer different questions.


## 9. Treemap styling tweaks — experiment with them

### Tweak 1: Change the colour palette

Colour can make patterns easier to see. A sequential palette is appropriate when a darker or stronger colour means a larger value.

Try changing `color_continuous_scale` to `"Viridis"`, `"Greens"`, `"Oranges"` or `"RdBu"` and compare the result.


In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    color="Sales_Lakh",
    color_continuous_scale="Viridis",
    title="Treemap with Viridis Colour Scale"
)
fig.show()


### Tweak 2: Colour by a different variable

Rectangle size can show sales while colour shows units sold. This allows two measures to be viewed together.

Be careful: colour and area now mean different things. Explain the colour scale in the chart.


In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",       # area = sales
    color="Units_Sold",        # colour = units sold
    color_continuous_scale="RdYlBu",
    title="Treemap: Area = Sales; Colour = Units Sold"
)
fig.update_layout(coloraxis_colorbar_title="Units sold")
fig.show()


### Tweak 3: Show only selected hierarchy levels

When there are many products, labels can become crowded. `maxdepth` limits the number of levels initially displayed.

Try:
- `maxdepth=1`: category level;
- `maxdepth=2`: category and subcategory levels;
- `maxdepth=-1`: show all levels.


In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    maxdepth=2,
    title="Treemap Limited to Category and Subcategory"
)
fig.show()


### Tweak 4: Change text and hover content

`textinfo` controls text drawn in each rectangle. Common options include:
- `"label"` — category or product name;
- `"label+value"` — name and value;
- `"label+percent parent"` — name and share of parent;
- `"label+value+percent parent"` — combines them.

Hover templates provide more detailed information without overcrowding the rectangles.


In [ ]:
fig = px.treemap(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    title="Treemap with Compact Labels"
)
fig.update_traces(
    textinfo="label+percent parent",
    hovertemplate="<b>%{label}</b><br>Value: %{value:.2f} lakh<extra></extra>"
)
fig.show()


### Tweak 5: Change the tiling/layout approach

Plotly's treemap trace supports tiling options such as:
- `squarify` — tries to create rectangles with more balanced aspect ratios;
- `binary` — recursively splits the space into two regions;
- `dice` and `slice` — split space in horizontal or vertical directions.

The underlying values and area proportions do not change; the arrangement and rectangle shapes can change.


In [ ]:
# Create a complete hierarchy
labels = ["Retailer"]
parents = [""]

values = [df["Sales_Lakh"].sum()]

# Add categories
for category in df["Category"].unique():
    labels.append(category)
    parents.append("Retailer")

    category_sales = df.loc[
        df["Category"] == category, "Sales_Lakh"
    ].sum()

    values.append(category_sales)

# Add subcategories
for subcategory in df["Subcategory"].unique():
    labels.append(subcategory)

    category = df.loc[
        df["Subcategory"] == subcategory, "Category"
    ].iloc[0]

    parents.append(category)

    subcategory_sales = df.loc[
        df["Subcategory"] == subcategory, "Sales_Lakh"
    ].sum()

    values.append(subcategory_sales)

# Add products
for _, row in df.iterrows():
    labels.append(row["Product"])
    parents.append(row["Subcategory"])
    values.append(row["Sales_Lakh"])

# Plot
fig = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    tiling=dict(packing="squarify")
))

fig.update_layout(
    title="Retail Sales Treemap – Squarified Layout",
    width=950,
    height=700
)

fig.show()

**Note on the tiling demonstration:** The preceding low-level example shows the product-to-subcategory layer. For a complete multi-level hierarchy, use the `path` approach shown earlier or provide a complete `labels` / `parents` tree. The `path` approach is usually simpler for a DataFrame.


## 10. How to verify area proportions from the data

A chart renderer does not expose rectangle pixel dimensions as a simple DataFrame column. But we can verify the **target area shares** mathematically from the values.

For siblings under the same parent:

\[
\text{Target area share}=
\frac{\text{Child value}}{\sum \text{Sibling values}}
\]

The following calculates each product's share within its subcategory.


In [ ]:
df["Share_of_Subcategory"] = (
    df["Sales_Lakh"] /
    df.groupby(["Category", "Subcategory"])["Sales_Lakh"].transform("sum")
)

df["Share_of_Total"] = df["Sales_Lakh"] / df["Sales_Lakh"].sum()

display(
    df[["Category", "Subcategory", "Product", "Sales_Lakh",
        "Share_of_Subcategory", "Share_of_Total"]]
      .sort_values("Sales_Lakh", ascending=False)
      .head(12)
)


,Category,Subcategory,Product,Sales_Lakh,Share_of_Subcategory,Share_of_Total
3,Electronics,Computers,Laptops,83.12,0.495972,0.046598
7,Electronics,Home Entertainment,Speakers,79.36,0.441822,0.044490
15,Fashion,Accessories,Watches,76.77,0.406944,0.043038
1,Electronics,Mobiles,Smartphone B,74.11,0.364356,0.041547
27,Sports,Fitness,Dumbbells,72.12,0.349893,0.040431
29,Sports,Fitness,Treadmills,72.08,0.349699,0.040409
12,Fashion,Women,Dresses,71.73,0.413668,0.040213
30,Sports,Outdoor,Camping Gear,69.97,0.435353,0.039226
16,Fashion,Accessories,Bags,67.94,0.360138,0.038088
0,Electronics,Mobiles,Smartphone A,67.59,0.332301,0.037892


**Interpretation:** If Product A contributes 40% of the sales in its subcategory, its rectangle should occupy 40% of that subcategory's treemap area (subject to the chart's layout and rendering). Do not compare its area with a product in another subcategory unless you use their share of the common root total.


# Part B — Sunburst charts

## 11. What is a sunburst chart?

A **sunburst chart** displays hierarchy as concentric rings:

- The centre represents the root.
- The first ring represents the root's children.
- The next ring represents grandchildren.
- Each additional ring represents another level.

The **angular span** of a segment is proportional to its value relative to the total represented by its parent. The radial thickness is generally shared by the ring level; it is not the value encoding.

A sunburst is particularly useful for showing how a hierarchy branches from a central root.


## 12. Create your first sunburst chart

In [ ]:
fig = px.sunburst(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    title="Retail Sales Sunburst: Category → Subcategory → Product"
)
fig.show()


### Interpret the rings

- Compare the angular spans of category segments to compare category totals.
- Within a category, compare subcategory segments to understand the category's composition.
- The outermost ring displays products.
- Click a segment to focus on that branch, then use the centre to move back up.

For example, if one subcategory occupies one quarter of its parent category's arc, it represents approximately 25% of that parent's value.


## 13. Sunburst styling tweaks

### Tweak 1: Colour by sales and customize hover text

In [ ]:
fig = px.sunburst(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    color="Sales_Lakh",
    color_continuous_scale="Plasma",
    title="Sunburst with Sales Colour Scale"
)
fig.update_traces(
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Sales: ₹%{value:.2f} lakh<br>"
        "Share of parent: %{percentParent:.1%}<br>"
        "Share of total: %{percentRoot:.1%}"
        "<extra></extra>"
    )
)
fig.update_layout(title_x=0.5, margin=dict(t=70, l=20, r=20, b=20))
fig.show()


### Tweak 2: Limit the visible depth

`maxdepth` controls how many rings are shown initially. This is helpful when the hierarchy has many levels or labels become too small.


In [ ]:
fig = px.sunburst(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    maxdepth=2,
    title="Sunburst Showing Two Levels"
)
fig.show()


### Tweak 3: Change the colour strategy

Colour can represent the category (discrete colours) rather than a continuous numerical scale. Here, the first-level category determines the colour family.


In [ ]:
fig = px.sunburst(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    color="Category",
    title="Sunburst Coloured by Top-Level Category"
)
fig.show()


### Tweak 4: Change the text shown on segments

In [ ]:
fig = px.sunburst(
    df,
    path=["Category", "Subcategory", "Product"],
    values="Sales_Lakh",
    title="Sunburst with Compact Text"
)
fig.update_traces(
    insidetextorientation="radial",
    textinfo="label+percent parent"
)
fig.show()


## 14. Treemap versus sunburst — same data, different visual encoding

| Question | Treemap | Sunburst |
|---|---|---|
| Which category contributes most? | Compare rectangle areas | Compare category arc spans |
| What is inside a category? | Nested rectangles | Outward rings |
| Which product is largest? | Compare product rectangle areas | Compare product angular spans within the parent |
| Is the hierarchy deep? | Deep nesting can become visually dense | Many rings can become difficult to read |
| Need efficient use of rectangular screen space? | Usually better | Not its main strength |

Neither chart is universally better. Choose based on the question, number of levels, number of nodes and available space.


# Part C — Practical interpretation and common mistakes

## 15. Common mistakes to avoid

1. **Comparing widths instead of areas:** In a treemap, area is the value encoding. Width alone is not.
2. **Comparing a child with an unrelated parent:** Compare siblings when interpreting share of parent.
3. **Confusing colour with size:** If colour encodes units and area encodes sales, they communicate different measures.
4. **Too many tiny labels:** Aggregate, filter or limit the displayed depth.
5. **Incorrect parent totals:** If using a pre-aggregated hierarchy, make sure parent values are consistent with child totals.
6. **Duplicate labels in a path:** If two branches contain identically named nodes, use unique identifiers or carefully construct the hierarchy so the paths remain unambiguous.
7. **Using a hierarchy chart for unrelated categories:** If there is no meaningful parent-child relationship, a bar chart may be more appropriate.

### `branchvalues`: an important Plotly detail

When building a treemap or sunburst directly with `go.Treemap` or `go.Sunburst`, `branchvalues` controls how parent values are interpreted:

- `"total"`: a parent value includes the values of its descendants. The parent total should equal the sum of its children (or be consistent with it).
- `"remainder"`: a parent value represents the amount remaining at the parent itself, in addition to its children.

With Plotly Express `path` and `values`, the hierarchy is aggregated from the leaf rows, so you usually do not need to set this manually.


## 16. Mini classroom activity

Use the generated `df` dataset.

**Task A — Treemap**
1. Create a treemap with Category, Subcategory and Product.
2. Make area represent `Units_Sold` instead of `Sales_Lakh`.
3. Make colour represent `Sales_Lakh`.
4. Display product labels and percentages.
5. Limit the initial view to two levels.

**Task B — Sunburst**
1. Create a sunburst with the same hierarchy.
2. Use `Sales_Lakh` for segment size.
3. Colour by `Category`.
4. Show the share of parent in the hover tooltip.
5. Compare the chart with the treemap.

**Task C — Interpretation**
1. Which category has the highest total sales?
2. Which subcategory has the highest total sales?
3. Which product contributes the largest share of total sales?
4. Find one product with a high number of units sold but relatively low sales. What might explain this?
5. Which chart communicates the hierarchy more clearly for this dataset? Justify your answer.


## 17. Solutions / checking your answers

In [ ]:
# 1. Category with highest sales
print("Top category:")
display(category_totals.head(1))

# 2. Subcategory with highest sales
print("Top subcategory:")
display(subcategory_totals.head(1))

# 3. Product with highest sales
print("Top product by sales:")
display(df.nlargest(1, "Sales_Lakh")[
    ["Category", "Subcategory", "Product", "Sales_Lakh", "Units_Sold"]
])

# 4. Compare average revenue per unit (₹ lakh per unit)
df["Sales_per_Unit_Lakh"] = df["Sales_Lakh"] / df["Units_Sold"]
print("Products with relatively high units but low sales per unit:")
display(df.sort_values("Sales_per_Unit_Lakh").head(5)[
    ["Product", "Sales_Lakh", "Units_Sold", "Sales_per_Unit_Lakh"]
])


## 18. Quick recap

- Hierarchical data has parent-child relationships and multiple levels.
- A treemap uses **nested rectangles**; rectangle **area** is proportional to the value.
- In a treemap, width and height are layout decisions; do not interpret either dimension alone.
- A sunburst uses **concentric rings**; segment angular span represents value relative to its parent.
- Both charts can show multiple levels of a hierarchy.
- Use colour, labels, hover details and depth controls to improve readability.
- Use a bar chart instead when precise comparison of values is more important than showing hierarchy.

### One-sentence takeaway

**Treemap = compare values through nested rectangle areas; Sunburst = explore parent-child structure through concentric rings.**
